title: Exploring SQLite with SQLAlchemy ORM and Automap

tags: #SQL #pandas #database #data_modeling #data_cleaning #analysis

category: experiments

aliases: [sqlalchemy-orm-sqlite]

In [5]:
# --- Imports ---
import pandas as pd
from sqlalchemy import create_engine, MetaData, Column, Integer, String, Numeric, Float, ForeignKey
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship, sessionmaker, Session

# %%
# --- Connect to SQLite Database ---
db_path = "../../Datasets/movies.db"
engine = create_engine(f"sqlite:///{db_path}", echo=False)

# === OPTION A: Automap ORM Classes ===

In [ ]:

metadata = MetaData()
metadata.reflect(bind=engine)

# Option: reflect with automap and display tables
AutoBase = automap_base(metadata=metadata)
AutoBase.prepare()

print("📋 Available Tables:", list(AutoBase.classes.keys()))

# Create session using automap
session = Session(bind=engine)

try:
    Movie = AutoBase.classes.movies
except AttributeError:
    raise ValueError("❌ 'movies' table not found in automap.")

print("🧬 Movie Columns:", [col for col in dir(Movie) if not col.startswith('_') and not col.startswith('metadata')])

In [3]:
# --- Automap Query Preview ---
results = session.query(Movie).limit(5).all()
for row in results:
    print(row.__dict__)

{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001389E50F290>, 'title': 'Inception', 'year': Decimal('2010.0000000000'), 'id': 1}
{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001389E50FFB0>, 'title': 'The Dark Knight', 'year': Decimal('2008.0000000000'), 'id': 2}
{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001389E5A8050>, 'title': 'Interstellar', 'year': Decimal('2014.0000000000'), 'id': 3}
{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001389E5A80B0>, 'title': 'Parasite', 'year': Decimal('2019.0000000000'), 'id': 4}
{'_sa_instance_state': <sqlalchemy.orm.state.InstanceState object at 0x000001389E5A8110>, 'title': 'The Matrix', 'year': Decimal('1999.0000000000'), 'id': 5}


In [4]:
# --- pandas Query ---
df_movies = pd.read_sql("SELECT * FROM movies LIMIT 5", con=engine)
print("📊 DataFrame Preview:")
print(df_movies)

📊 DataFrame Preview:
   id            title  year
0   1        Inception  2010
1   2  The Dark Knight  2008
2   3     Interstellar  2014
3   4         Parasite  2019
4   5       The Matrix  1999


# === OPTION B: Define ORM Models Manually ===

In [6]:
Base = declarative_base()

# Need to ensure autoincrement for primary key
# and foreign key as assigned.

class Movie(Base):
    __tablename__ = 'movies'
    id = Column(Integer, primary_key=True,autoincrement=True)
    title = Column(String)
    year = Column(Numeric)
    ratings = relationship("Rating", back_populates="movie")
    stars = relationship("Star", back_populates="movie")

class Person(Base):
    __tablename__ = 'people'
    id = Column(Integer, primary_key=True,autoincrement=True)
    name = Column(String)
    birth = Column(Numeric)
    stars = relationship("Star", back_populates="person")

class Rating(Base):
    __tablename__ = 'ratings'
    id = Column(Integer, primary_key=True)
    movie_id = Column(Integer, ForeignKey('movies.id'))
    rating = Column(Float)
    votes = Column(Integer)
    movie = relationship("Movie", back_populates="ratings")

class Star(Base):
    __tablename__ = 'stars'
    movie_id = Column(Integer, ForeignKey('movies.id'), primary_key=True)
    person_id = Column(Integer, ForeignKey('people.id'), primary_key=True)
    movie = relationship("Movie", back_populates="stars")
    person = relationship("Person", back_populates="stars")

# --- Start Session for Manual Base ---
Session = sessionmaker(bind=engine)
session = Session()

C:\Users\RhysL\AppData\Local\Temp\ipykernel_18104\2789458597.py:1: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [12]:
# --- Create Function ---
def create_entry(entry):
    """
    Adds a new record to the database and commits the transaction.

    Parameters:
    entry (Base): An instance of a mapped SQLAlchemy model.
    """
    try:
        session.add(entry)
        session.commit()
        print(f"✅ Entry added: {entry}")
    except Exception as e:
        session.rollback()
        print("❌ Failed to add entry:", e)

# --- Update Function ---
def update_entry():
    """
    Commits the current state of modified objects to the database.
    Call this after changing attributes of an existing object.
    """
    try:
        session.commit()
        print("🛠️ Update committed.")
    except Exception as e:
        session.rollback()
        print("❌ Failed to update entry:", e)

# --- Delete Function ---
def delete_entry(entry):
    """
    Deletes the specified record from the database and commits the transaction.

    Parameters:
    entry (Base): An instance of a mapped SQLAlchemy model.
    """
    try:
        session.delete(entry)
        session.commit()
        print(f"🗑️ Deleted entry: {entry}")
    except Exception as e:
        session.rollback()
        print("❌ Failed to delete entry:", e)


In [9]:
# --- Create a New Movie ---
new_movie = Movie(title="New Film", year=20111)
create_entry(new_movie)

✅ Entry added: <__main__.Movie object at 0x000001389E53B830>


In [10]:
# --- Update That Movie ---
movie = session.query(Movie).filter_by(title="New Film").first()
if movie:
    movie.year = 999
    update_entry()

🛠️ Update committed.


In [26]:
movie = session.query(Movie).filter_by(title="Inception").first()
movie.year

Decimal('2010.0000000000')

In [17]:
# --- Find and Delete a Movie ---
movie = session.query(Movie).filter_by(title="New Film").first()
if movie:
    delete_entry(movie)

In [ ]:
# --- Transaction with Rollback ---
try:
    with session.begin():
        movie1 = Movie(title="Tx Test 1", year=2030)
        movie2 = Movie(title="Tx Test 2", year=2031)
        session.add_all([movie1, movie2])
        # raise Exception("Force rollback test")
    print("✅ Transaction committed.")
except Exception as e:
    print("❌ Rolled back due to:", e)

✅ Transaction committed.


In [ ]:
# --- Join Example: Movies + Ratings (Manual ORM) ---
joined = (
    session.query(Movie.title, Rating.rating)
    .join(Rating, Movie.id == Rating.movie_id)
    .limit(5)
    .all()
)
print("🔗 Joined movies with ratings:")
for row in joined:
    print(row)

🔗 Joined movies with ratings:
('Inception', 8.8)
('The Dark Knight', 9.0)
('Interstellar', 8.6)
('Parasite', 8.5)
('The Matrix', 8.7)


In [18]:
# --- Query: Get Votes from Ratings for a Given Movie ---
def get_movie_votes(title):
    query = (
        session.query(Movie.title, Rating.votes)
        .join(Rating, Movie.id == Rating.movie_id)
        .filter(Movie.title == title)
        .all()
    )
    return pd.DataFrame([{"title": t, "votes": v} for t, v in query])

print("\nVotes for 'The Matrix':")
print(get_movie_votes("The Matrix"))


Votes for 'The Matrix':
        title    votes
0  The Matrix  1800000


In [19]:
# --- Query: Get People Who Starred in Each Movie ---
def get_movie_stars(title):
    query = (
        session.query(Movie.title, Person.name)
        .join(Star, Movie.id == Star.movie_id)
        .join(Person, Star.person_id == Person.id)
        .filter(Movie.title == title)
        .all()
    )
    return pd.DataFrame([{"title": t, "person": p} for t, p in query])

print("\nPeople who starred in 'The Matrix':")
print(get_movie_stars("The Matrix"))



People who starred in 'The Matrix':
        title        person
0  The Matrix  Keanu Reeves


In [21]:
# --- Load and Execute SQL Query from File ---
query_file = "Q1.txt"
try:
    with open(query_file, "r") as file:
        query = file.read().strip()

    print(f"\nExecuting Query from {query_file}:\n{query}")
    df_query_result = pd.read_sql_query(query, con=engine)
    print("\nQuery Result:")
    print(df_query_result)

except FileNotFoundError:
    print(f"\nError: Query file '{query_file}' not found.")
except Exception as e:
    print(f"\n❌ Failed to execute query: {e}")



Executing Query from Q1.txt:
SELECT id, title, year AS release_year
FROM movies
WHERE year > 2010
ORDER BY title ASC;

Query Result:
   id         title  release_year
0   3  Interstellar          2014
1   4      Parasite          2019
2  11     Tx Test 1          2030
3  12     Tx Test 2          2031


In [ ]:
# --- Close Session ---
session.close()
print("🔒 Session closed.")

🔒 Session closed.
